# Pacotes

In [1]:
import os
import sys
import yaml
import pandas as pd

current_dir = os.path.dirname(os.path.abspath('.'))
project_root = os.path.abspath(os.path.join(current_dir, "../.."))
sys.path.insert(0, project_root)

from functions.feature_selection import FeatureSelectionOrchestrator
from utils.plots import Pearson_correlation, Bar_plot
from Regression.house_prices.src.features.feat_eng_pipeline import PreprocessingOrchestrator

In [2]:
import warnings

warnings.filterwarnings('ignore')

# Config

In [3]:
with open(os.path.join("../config/config.yaml"), "r") as f:
    config = yaml.safe_load(f)
        
with open(os.path.join( "../config/pipeline.yaml"), "r") as f:
    config_pipe = yaml.safe_load(f)  

# Load dataset - preprocessing

In [4]:
X_train = pd.read_parquet(
        os.path.join(
            config['init_path'],
            config['data']['processed'],
            "train_features.parquet")
    )
y_train = X_train[config_pipe['features']['preprocessing']['target'][0]] 


In [5]:
preprocessor = PreprocessingOrchestrator(
        numerical_con_1=config_pipe['features']['preprocessing']['num_con_1'], 
        numerical_con_2=None,
        numerical_dis=config_pipe['features']['preprocessing']['num_dis_1'], 
        categorical_var=config_pipe['features']['preprocessing']['cat_var']
        )

In [6]:
pipe = preprocessor.apply("preprocessing")    
X_train_trans = pipe.fit_transform(X_train)

In [7]:
X_train_trans

,numerical_con_pipe__LotArea,numerical_con_pipe__MasVnrArea,numerical_con_pipe__BsmtFinSF1,numerical_con_pipe__BsmtFinSF2,numerical_con_pipe__BsmtUnfSF,numerical_con_pipe__TotalBsmtSF,numerical_con_pipe__1stFlrSF,numerical_con_pipe__2ndFlrSF,numerical_con_pipe__GrLivArea,numerical_con_pipe__GarageArea,...,categorical_pipe__GarageType,categorical_pipe__GarageFinish,categorical_pipe__GarageQual,categorical_pipe__GarageCond,categorical_pipe__PavedDrive,categorical_pipe__PoolQC,categorical_pipe__Fence,categorical_pipe__MiscFeature,categorical_pipe__SaleType,categorical_pipe__SaleCondition
0,8450,196.0,706,0,150,856,856,854,1710,548,...,Attchd,RFn,TA,TA,Y,missing,missing,missing,WD,Normal
1,9600,0.0,978,0,284,1262,1262,0,1262,460,...,Attchd,RFn,TA,TA,Y,missing,missing,missing,WD,Normal
2,11250,162.0,486,0,434,920,920,866,1786,608,...,Attchd,RFn,TA,TA,Y,missing,missing,missing,WD,Normal
3,9550,0.0,216,0,540,756,961,756,1717,642,...,Detchd,Unf,TA,TA,Y,missing,missing,missing,WD,Abnorml
4,14260,350.0,655,0,490,1145,1145,1053,2198,836,...,Attchd,RFn,TA,TA,Y,missing,missing,missing,WD,Normal
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1455,7917,0.0,0,0,953,953,953,694,1647,460,...,Attchd,RFn,TA,TA,Y,missing,missing,missing,WD,Normal
1456,13175,119.0,790,163,589,1542,2073,0,2073,500,...,Attchd,Unf,TA,TA,Y,missing,MnPrv,missing,WD,Normal
1457,9042,0.0,275,0,877,1152,1188,1152,2340,252,...,Attchd,RFn,TA,TA,Y,missing,GdPrv,Shed,WD,Normal
1458,9717,0.0,49,1029,0,1078,1078,0,1078,240,...,Attchd,Unf,TA,TA,Y,missing,missing,missing,WD,Normal


# Univariate Feat Selection

In [8]:
feature_selection = FeatureSelectionOrchestrator()

In [9]:
constant_features = feature_selection.apply(
        "DropConstantFeatures", 
        X_train_trans, 
        y_train,
        threshold=0.95)

In [10]:
constant_features

['numerical_dis_pipe__LowQualFinSF',
 'numerical_dis_pipe__KitchenAbvGr',
 'numerical_dis_pipe__3SsnPorch',
 'numerical_dis_pipe__PoolArea',
 'numerical_dis_pipe__MiscVal',
 'categorical_pipe__Street',
 'categorical_pipe__Utilities',
 'categorical_pipe__Condition2',
 'categorical_pipe__RoofMatl',
 'categorical_pipe__Heating',
 'categorical_pipe__PoolQC',
 'categorical_pipe__MiscFeature']

In [11]:
duplicate_features = feature_selection.apply(
        "DropDuplicateFeatures", 
        X_train_trans,        
        y_train
        )

In [12]:
duplicate_features

set()

In [13]:
QuiSquare = feature_selection.apply(
        "QuiSquare", 
        X_train_trans.filter(like='categorical'), 
        y_train)

Feature: categorical_pipe__MSZoning
Feature: categorical_pipe__Street
Feature: categorical_pipe__Alley
Feature: categorical_pipe__LotShape
Feature: categorical_pipe__LandContour
Feature: categorical_pipe__Utilities
Feature: categorical_pipe__LotConfig
Feature: categorical_pipe__LandSlope
Feature: categorical_pipe__Neighborhood
Feature: categorical_pipe__Condition1
Feature: categorical_pipe__Condition2
Feature: categorical_pipe__BldgType
Feature: categorical_pipe__HouseStyle
Feature: categorical_pipe__RoofStyle
Feature: categorical_pipe__RoofMatl
Feature: categorical_pipe__Exterior1st
Feature: categorical_pipe__Exterior2nd
Feature: categorical_pipe__MasVnrType
Feature: categorical_pipe__ExterQual
Feature: categorical_pipe__ExterCond
Feature: categorical_pipe__Foundation
Feature: categorical_pipe__BsmtQual
Feature: categorical_pipe__BsmtCond
Feature: categorical_pipe__BsmtExposure
Feature: categorical_pipe__BsmtFinType1
Feature: categorical_pipe__BsmtFinType2
Feature: categorical_pipe__H

In [14]:
# feature selection based on QuiSquare test
[i
    for i in QuiSquare.loc[QuiSquare > 0.05].index
]

['categorical_pipe__GarageQual',
 'categorical_pipe__GarageType',
 'categorical_pipe__Condition2',
 'categorical_pipe__LandContour',
 'categorical_pipe__LandSlope',
 'categorical_pipe__BsmtFinType1',
 'categorical_pipe__HouseStyle',
 'categorical_pipe__Electrical',
 'categorical_pipe__Exterior2nd',
 'categorical_pipe__BsmtFinType2',
 'categorical_pipe__Alley',
 'categorical_pipe__PavedDrive',
 'categorical_pipe__HeatingQC',
 'categorical_pipe__Exterior1st',
 'categorical_pipe__BldgType',
 'categorical_pipe__PoolQC',
 'categorical_pipe__GarageCond',
 'categorical_pipe__Fence',
 'categorical_pipe__MiscFeature',
 'categorical_pipe__Functional',
 'categorical_pipe__Utilities',
 'categorical_pipe__Condition1',
 'categorical_pipe__RoofStyle',
 'categorical_pipe__RoofMatl']

In [15]:
# feature selection based on QuiSquare test
[i
    for i in QuiSquare.loc[QuiSquare < 0.05].index
]

['categorical_pipe__ExterQual',
 'categorical_pipe__KitchenQual',
 'categorical_pipe__BsmtQual',
 'categorical_pipe__Heating',
 'categorical_pipe__BsmtCond',
 'categorical_pipe__BsmtExposure',
 'categorical_pipe__SaleType',
 'categorical_pipe__SaleCondition',
 'categorical_pipe__ExterCond',
 'categorical_pipe__LotShape',
 'categorical_pipe__MSZoning',
 'categorical_pipe__GarageFinish',
 'categorical_pipe__Street',
 'categorical_pipe__Neighborhood',
 'categorical_pipe__FireplaceQu',
 'categorical_pipe__MasVnrType',
 'categorical_pipe__Foundation',
 'categorical_pipe__CentralAir',
 'categorical_pipe__LotConfig']

In [16]:
Anova = feature_selection.apply(
        "Anova",
        X_train_trans.filter(like='numerical'),
        y_train)
    
mi = feature_selection.apply(
        "MutualInformationClassif", 
        X_train_trans.filter(like='numerical'), 
        y_train)
    
corr = feature_selection.apply(
        "PearsonCorrelation", 
        X_train_trans.filter(like='numerical'), 
        y_train)

In [17]:
[i.replace("numerical_dis_pipe__", "")
    for i in 
    Anova.loc[Anova > 0.05].index
]

['OverallCond',
 'YrSold',
 'numerical_con_pipe__ScreenPorch',
 'numerical_con_pipe__BsmtFinSF2',
 'PoolArea',
 'MoSold',
 '3SsnPorch',
 'KitchenAbvGr',
 'MSSubClass',
 'LowQualFinSF',
 'BsmtHalfBath',
 'numerical_con_pipe__EnclosedPorch']

In [18]:
[i.replace("numerical_dis_pipe__", "")
    for i in 
    Anova.loc[Anova < 0.05].index
]

['OverallQual',
 'MiscVal',
 'numerical_con_pipe__GrLivArea',
 'numerical_con_pipe__LotArea',
 'GarageCars',
 'FullBath',
 'numerical_con_pipe__GarageArea',
 'numerical_con_pipe__1stFlrSF',
 'numerical_con_pipe__TotalBsmtSF',
 'YearBuilt',
 'numerical_con_pipe__MasVnrArea',
 'TotRmsAbvGrd',
 'YearRemodAdd',
 'GarageYrBlt',
 'numerical_con_pipe__BsmtFinSF1',
 'numerical_con_pipe__2ndFlrSF',
 'Fireplaces',
 'numerical_con_pipe__BsmtUnfSF',
 'numerical_con_pipe__OpenPorchSF',
 'HalfBath',
 'numerical_con_pipe__WoodDeckSF',
 'BedroomAbvGr',
 'LotFrontage',
 'BsmtFullBath']

In [19]:
[i.replace("numerical_con_pipe__", "")
    for i in 
    Anova.loc[Anova > 0.05].index
]

['numerical_dis_pipe__OverallCond',
 'numerical_dis_pipe__YrSold',
 'ScreenPorch',
 'BsmtFinSF2',
 'numerical_dis_pipe__PoolArea',
 'numerical_dis_pipe__MoSold',
 'numerical_dis_pipe__3SsnPorch',
 'numerical_dis_pipe__KitchenAbvGr',
 'numerical_dis_pipe__MSSubClass',
 'numerical_dis_pipe__LowQualFinSF',
 'numerical_dis_pipe__BsmtHalfBath',
 'EnclosedPorch']

In [20]:
[i.replace("numerical_con_pipe__", "")
    for i in 
    Anova.loc[Anova < 0.05].index
]

['numerical_dis_pipe__OverallQual',
 'numerical_dis_pipe__MiscVal',
 'GrLivArea',
 'LotArea',
 'numerical_dis_pipe__GarageCars',
 'numerical_dis_pipe__FullBath',
 'GarageArea',
 '1stFlrSF',
 'TotalBsmtSF',
 'numerical_dis_pipe__YearBuilt',
 'MasVnrArea',
 'numerical_dis_pipe__TotRmsAbvGrd',
 'numerical_dis_pipe__YearRemodAdd',
 'numerical_dis_pipe__GarageYrBlt',
 'BsmtFinSF1',
 '2ndFlrSF',
 'numerical_dis_pipe__Fireplaces',
 'BsmtUnfSF',
 'OpenPorchSF',
 'numerical_dis_pipe__HalfBath',
 'WoodDeckSF',
 'numerical_dis_pipe__BedroomAbvGr',
 'numerical_dis_pipe__LotFrontage',
 'numerical_dis_pipe__BsmtFullBath']

In [21]:
smart_corr = feature_selection.apply(
        "SmartCorrelatedSelection",
        X_train_trans.filter(like='numerical'),
        y_train)

In [22]:
smart_corr

{'corr_feature': ['numerical_con_pipe__1stFlrSF',
  'numerical_con_pipe__TotalBsmtSF'],
 'corr_2_drop': ['numerical_con_pipe__TotalBsmtSF',
  'numerical_dis_pipe__GarageCars',
  'numerical_dis_pipe__TotRmsAbvGrd']}

In [23]:
corr.loc['numerical_dis_pipe__TotRmsAbvGrd']

numerical_con_pipe__LotArea          0.190015
numerical_con_pipe__MasVnrArea       0.279568
numerical_con_pipe__BsmtFinSF1       0.044316
numerical_con_pipe__BsmtFinSF2      -0.035227
numerical_con_pipe__BsmtUnfSF        0.250647
numerical_con_pipe__TotalBsmtSF      0.285573
numerical_con_pipe__1stFlrSF         0.409516
numerical_con_pipe__2ndFlrSF         0.616423
numerical_con_pipe__GrLivArea        0.825489
numerical_con_pipe__GarageArea       0.337822
numerical_con_pipe__WoodDeckSF       0.165984
numerical_con_pipe__OpenPorchSF      0.234192
numerical_con_pipe__EnclosedPorch    0.004151
numerical_con_pipe__ScreenPorch      0.059383
numerical_dis_pipe__MSSubClass       0.040380
numerical_dis_pipe__LotFrontage      0.320518
numerical_dis_pipe__OverallQual      0.427452
numerical_dis_pipe__OverallCond     -0.057583
numerical_dis_pipe__YearBuilt        0.095589
numerical_dis_pipe__YearRemodAdd     0.191740
numerical_dis_pipe__LowQualFinSF     0.131185
numerical_dis_pipe__BsmtFullBath  

In [24]:
mrmr = feature_selection.apply(
        "MRMR",        
        X_train_trans,
        y_train,
        method="MIQ"
        )

In [25]:
mrmr['features_to_drop']

['numerical_con_pipe__LotArea',
 'numerical_con_pipe__MasVnrArea',
 'numerical_con_pipe__BsmtFinSF1',
 'numerical_con_pipe__BsmtUnfSF',
 'numerical_con_pipe__TotalBsmtSF',
 'numerical_con_pipe__1stFlrSF',
 'numerical_con_pipe__2ndFlrSF',
 'numerical_con_pipe__GrLivArea',
 'numerical_con_pipe__GarageArea',
 'numerical_con_pipe__WoodDeckSF',
 'numerical_con_pipe__OpenPorchSF',
 'numerical_con_pipe__EnclosedPorch',
 'numerical_dis_pipe__MSSubClass',
 'numerical_dis_pipe__LotFrontage',
 'numerical_dis_pipe__OverallQual',
 'numerical_dis_pipe__OverallCond',
 'numerical_dis_pipe__YearBuilt',
 'numerical_dis_pipe__YearRemodAdd',
 'numerical_dis_pipe__LowQualFinSF',
 'numerical_dis_pipe__BsmtFullBath',
 'numerical_dis_pipe__BsmtHalfBath',
 'numerical_dis_pipe__FullBath',
 'numerical_dis_pipe__HalfBath',
 'numerical_dis_pipe__BedroomAbvGr',
 'numerical_dis_pipe__Fireplaces',
 'numerical_dis_pipe__GarageYrBlt',
 'numerical_dis_pipe__3SsnPorch',
 'numerical_dis_pipe__PoolArea',
 'numerical_dis_pi

In [26]:
mrmr['relevance']

array([0.13800111, 0.11831558, 0.09277383, 0.        , 0.0146596 ,
       0.23133701, 0.15994908, 0.13051426, 0.38510906, 0.29261043,
       0.09802921, 0.15214871, 0.02504097, 0.        , 0.3438294 ,
       0.2000799 , 1.05911094, 0.66996644, 0.29158726, 0.24046213,
       0.04906436, 0.37628391, 0.        , 2.05663156, 0.48020886,
       0.93706895, 2.74888927, 0.56767891, 0.81184076, 0.20728631,
       1.70275035, 0.01899561, 0.        , 0.        , 0.01314789,
       0.20999541])

In [ ]:
mrmr